# Data Leakage Detection at Scale

This notebook demonstrates advanced techniques for detecting data leakage in large datasets. We progress through four approaches, each suited to a different scale:

1. **Polars LazyFrames** -- scan and filter without loading everything into RAM
2. **Arrow C Data Interface** -- zero-copy memory handoff to native (C/C++/Rust) backends
3. **DataFusion SQL Engine** -- SQL-based detection directly on Parquet files
4. **Apache Iceberg** -- conceptual integration for versioned lakehouse tables

Compared to the pandas + scikit-learn approach in the previous notebook, these tools handle datasets that don't fit in memory and integrate with production data infrastructure.

In [ ]:
# Install dependencies if running in Colab
!pip install polars pyarrow datafusion --quiet

## Generate Synthetic Data

We create train and test Parquet files with deliberate leakage (shared rows across the split boundary). This makes the notebook fully self-contained -- no external files needed.

In [ ]:
import polars as pl
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
n_rows = 50_000

# Generate a base dataset with user activity data
base = pl.DataFrame({
    "id": range(n_rows),
    "user_id": np.random.randint(1, 5000, size=n_rows),
    "timestamp": [
        datetime(2024, 1, 1) + timedelta(hours=int(h))
        for h in np.random.randint(0, 8760, size=n_rows)
    ],
    "feat_0": np.random.randn(n_rows),
    "feat_1": np.random.randn(n_rows),
    "feat_2": np.random.randn(n_rows),
    "feat_3": np.random.randn(n_rows),
    "target": np.random.randint(0, 2, size=n_rows),
})

# Split 80/20
train = base[:40_000]
test = base[40_000:]

# INJECT LEAKAGE: copy 200 rows from train into test
leaked_rows = train[:200]
test_with_leakage = pl.concat([test, leaked_rows])

# Write to Parquet
train.write_parquet("train.parquet")
test_with_leakage.write_parquet("test.parquet")

print(f"Train: {train.shape[0]:,} rows")
print(f"Test:  {test_with_leakage.shape[0]:,} rows (includes 200 leaked from train)")

## Step 1: Scanning with Polars LazyFrames

Polars LazyFrames let us define a query plan without loading the full dataset into memory. The engine pushes predicates down to the Parquet reader, so only the columns and rows we actually need are ever materialized.

Here we use a `semi_join` to find rows in the test set whose `id` also appears in the training set -- a direct indicator of entity leakage.

In [ ]:
# Scan parquet files lazily -- nothing is loaded into RAM yet
lf_train = pl.scan_parquet("train.parquet")
lf_test = pl.scan_parquet("test.parquet")

# Semi-join: keep only test rows whose id exists in train (= leaked rows)
leaked = (
    lf_test
    .join(lf_train, on="id", how="semi")
    .collect()
)

print(f"Leaked rows detected via Polars semi-join: {leaked.shape[0]}")
print(f"\nFirst 5 leaked rows:")
print(leaked.head())

## Step 2: Zero-Copy Handoff via Arrow C Data Interface

The [Arrow C Data Interface](https://arrow.apache.org/docs/format/CDataInterface.html) enables zero-copy data sharing between runtimes. Instead of serializing data, we pass raw memory pointers -- the receiving C/C++/Rust library reads the same bytes without any copying.

This is the foundation for calling high-performance native validation libraries from Python. Below we demonstrate the mechanics: exporting a Polars-collected Arrow table to C-compatible pointers.

In [ ]:
import pyarrow as pa
import ctypes

# Collect the columns we need and convert Polars -> Arrow (zero-copy)
table = (
    pl.scan_parquet("train.parquet")
    .select(["timestamp", "user_id"])
    .collect()
    .to_arrow()
)

# Get the first RecordBatch
batch = table.to_batches()[0]


def export_to_c_pointers(batch: pa.RecordBatch):
    """
    Export an Arrow RecordBatch to C Data Interface pointers.
    In production, these pointers would be passed to a compiled
    C/C++/Rust library (e.g. a leakage validator) for zero-copy processing.
    """
    # Allocate C structs for the Arrow C Data Interface
    c_array = ctypes.c_int64(0)
    c_schema = ctypes.c_int64(0)

    # Export -- this does NOT copy data, it shares the memory address
    batch._export_to_c(ctypes.addressof(c_array), ctypes.addressof(c_schema))

    print(f"RecordBatch rows:  {batch.num_rows:,}")
    print(f"Array pointer:     {hex(c_array.value)}")
    print(f"Schema pointer:    {hex(c_schema.value)}")

    # --- HYPOTHETICAL NATIVE CALL ---
    # In production you would load a shared library and pass the pointers:
    #
    #   leakage_lib = ctypes.CDLL("./leakage_validator.so")
    #   result = leakage_lib.validate_temporal_integrity(
    #       ctypes.addressof(c_array), ctypes.addressof(c_schema)
    #   )
    #
    # The native code reads the Arrow memory directly -- no serialization,
    # no Python overhead on the hot path.
    # ---------------------------------

    return "Pointers exported successfully. Ready for native backend."


print(export_to_c_pointers(batch))

## Step 3: SQL-Based Detection with DataFusion

[DataFusion](https://datafusion.apache.org/) is an extensible SQL query engine built on Arrow. It can register Parquet files as tables and run SQL queries directly on them -- no need to load everything into a DataFrame first.

This is particularly powerful for leakage detection: we can express the check as a simple `EXISTS` subquery, and DataFusion handles predicate pushdown and parallelism under the hood.

In [ ]:
import datafusion

ctx = datafusion.SessionContext()

# Register Parquet files as SQL tables (no eager loading into RAM)
ctx.register_parquet("train", "train.parquet")
ctx.register_parquet("test", "test.parquet")

# Find leaked rows: test rows whose id also exists in train
leak_query = """
    SELECT test.*
    FROM test
    WHERE EXISTS (
        SELECT 1 FROM train WHERE train.id = test.id
    )
"""

leaked_df = ctx.sql(leak_query).to_pandas()
print(f"Leaked rows detected via DataFusion SQL: {len(leaked_df)}")
print(f"\nFirst 5 leaked rows:")
print(leaked_df.head())

# Count query for quick summary
count_result = ctx.sql("""
    SELECT
        (SELECT COUNT(*) FROM test) AS total_test_rows,
        (SELECT COUNT(*) FROM test
         WHERE EXISTS (SELECT 1 FROM train WHERE train.id = test.id)
        ) AS leaked_rows
""").to_pandas()

pct = (count_result["leaked_rows"][0] / count_result["total_test_rows"][0]) * 100
print(f"\nLeakage rate: {pct:.1f}% of test set")

## Step 4: Iceberg Integration (Conceptual)

[Apache Iceberg](https://iceberg.apache.org/) adds table-level versioning and time-travel to data lakes. In production, your train/test splits often live as partitions or snapshots inside an Iceberg table.

The function below sketches how you would wire DataFusion to an Iceberg catalog. This is **conceptual** -- Iceberg requires a catalog service (e.g. Hive Metastore, AWS Glue, or Nessie) that isn't available in Colab. In practice, you would use [PyIceberg](https://py.iceberg.apache.org/) to resolve the table metadata, then register the underlying Parquet files with DataFusion.

In [ ]:
def detect_leakage_iceberg(catalog_uri: str, table_name: str, id_column: str = "id"):
    """
    Conceptual: detect leakage across Iceberg table snapshots.

    In a real deployment this would:
    1. Connect to an Iceberg catalog (Hive, Glue, Nessie)
    2. Load the table via PyIceberg to resolve snapshot metadata
    3. Register the underlying Parquet files with DataFusion
    4. Run the same EXISTS query from Step 3

    Parameters
    ----------
    catalog_uri : str
        URI of the Iceberg catalog (e.g. "thrift://metastore:9083")
    table_name : str
        Fully qualified table name (e.g. "ml_pipeline.train_test_splits")
    id_column : str
        Column used to detect entity overlap
    """
    # --- PSEUDOCODE ---
    # from pyiceberg.catalog import load_catalog
    #
    # catalog = load_catalog("default", uri=catalog_uri)
    # table = catalog.load_table(table_name)
    #
    # # Get Parquet file paths from the current snapshot
    # scan = table.scan(row_filter=f"split IN ('train', 'test')")
    # parquet_files = [task.file.file_path for task in scan.plan_files()]
    #
    # ctx = datafusion.SessionContext()
    # for f in parquet_files:
    #     if "train" in f:
    #         ctx.register_parquet("train", f)
    #     else:
    #         ctx.register_parquet("test", f)
    #
    # return ctx.sql(f"""
    #     SELECT COUNT(*) AS leaked_rows
    #     FROM test
    #     WHERE EXISTS (
    #         SELECT 1 FROM train WHERE train.{id_column} = test.{id_column}
    #     )
    # """).collect()
    # ------------------

    return "Iceberg integration requires a catalog service. See PyIceberg docs."


# This won't connect to anything, but shows the intended API
print(detect_leakage_iceberg(
    catalog_uri="thrift://metastore:9083",
    table_name="ml_pipeline.splits",
))

## Summary

| Approach | Best For | Memory Model | Runs in Colab |
|:---|:---|:---|:---|
| **Polars LazyFrames** | Single-machine scans up to ~100 GB | Lazy, columnar, streams from Parquet | Yes |
| **Arrow C Data Interface** | Handing off to C/C++/Rust validators | Zero-copy pointer sharing | Partial (demo only) |
| **DataFusion SQL** | SQL-native teams, multi-file joins | Lazy, Arrow-backed execution | Yes |
| **Iceberg + DataFusion** | Versioned lakehouse tables | Snapshot-based, catalog-driven | No (needs catalog) |

The canonical pandas + scikit-learn approach from `01_startup_data_leakage.ipynb` works well for datasets that fit in RAM. When your data outgrows a single DataFrame, the tools above let you keep the same leakage checks without rewriting your logic -- just change the execution engine.